In [ ]:
# Python Ver.: 3.8.1
# Code Ver  : 1.3
# Date : 2021/04/06 19:44

import serial
import sys,time, datetime,random

class MA08(object):

	def __init__(self,com="COM5",baudrate=19200,at=30.00,avc=32,avmode="Moving"):
		
		self.com = com
		self.baudrate = baudrate
		self.connect()
		self.set_at(at)
		self.set_average_mode(avmode)
		self.set_average_count(avc)
		self.start()
		
	def connect_Quick(self):
		self.ser = serial.Serial(
				port = self.com,
				baudrate = self.baudrate,
				bytesize=7,
				parity=serial.PARITY_NONE,
				stopbits=1,				
				timeout=0.08,
				writeTimeout=0.1)
		self.ser.close()
		return True
		
	def connect(self):
		self.ser = serial.Serial(
				port = self.com,
				baudrate = self.baudrate,
				bytesize=7,
				parity=serial.PARITY_NONE,
				stopbits=1,				
				timeout=1.0,
				writeTimeout=1.0)
		self.ser.close()
		return True
	
	
	def close(self):
		self.ser.close()
		return True

	def read(self,byte = 1024):
		self.ser.open()
		res = self.ser.read(byte).decode()
		self.ser.close()
		return res
		
	def write(self,cmd):
		self.ser.open()
		self.ser.write(cmd.encode())
		self.ser.close()
		return True		

	def query(self,cmd,byte=1024):
		self.ser.open()
		self.ser.write(cmd.encode())
		res = self.ser.read(byte).decode()
		self.ser.close()
		return res
	
	def ID(self):
		cmd = "IDN?\n"
		res = self.query(cmd)
		return res
		
	def ZERO(self):
		cmd = "ZERO\n"
		i = 0
		print ("\n Power Sensor ZERO calibration start. ")
		self.write(cmd)
		print("  0.0 [%]",end="")
		while i < 20:
			i += 1
			data = i*5.0 + random.uniform(-1,1)
			if data > 100.0 : data = 100.0
			print("\r %.1f [%%]"%data,end="")
			time.sleep(1.0)
		print("\r 100.0 [%]")
		print(" ZERO Finish.\n")
		return True
	
	def start(self):
		cmd = "START\n"
		self.write(cmd)
		return True
		
	def stop(self):
		cmd = "STOP\n"
		self.write(cmd)
		return True		

	
	def meas(self):
		cmd = "PWR?\n"
		pwr = self.query(cmd)
		return pwr

	def show_power(self):
		pwr = self.meas()
		print (" Current Power = %.2f [dBm] "%float(pwr))
		return True
		
	def check_cf(self):
		cmd = "FREQ?\n"
		cf = self.query(cmd)
		return cf
		
	def set_cf(self,cf = 6.0):
		cmd = "FREQ %.3f\n"%cf
		res = self.query(cmd)
		return res
		
	def show_cf(self):
		cf = self.check_cf()
		print( "Current Centor Freq. = %.2f [GHz] "%float(cf))
		return True
		
	def set_at(self,at=30.00):
		if at >=300:
			print("平均パワー測定の有効なキャプチャー時間が大きすぎます。引数 at は 300(ms) 以下で設定してください。")
			sys.exit("chapert1")
		elif at < 0.01 :
			print("平均パワー測定の有効なキャプチャー時間が小さすぎます。引数 at は 0.01(ms) 以上で設定してください。")
		cmd = "CHAPERT %.2f\n"%at
		res=self.query(cmd)
		if res == "OK\n":pass
		else : 
			print("平均パワー測定の有効なキャプチャー時間を設定できませんでした。引数 at が長すぎる可能性があります。 at は 30 以下が推奨されます")
			sys.exit("chpaert2")
		return True

	def show_at(self):
		cmd = "CHAPERT?\n"
		res = self.query(cmd).rstrip("\n")
		print("Aperture time = "+res+"　[ms]")
		return True	
		
	def set_average_mode(self,mode="Moving"):
		if mode == "Moving":
			cmd = "AVGTYP 0\n"
		elif mode ==1 or mode == "Repeat":
			cmd = "AVGTYP 1\n"
		else:
			print("パワーセンサの平均化方式の指定が不正です。 引数 mode は Moving ( 移動法 )、または Repeat ( 繰り返し方 ) のどちらかを指定して下さい。")
			sys.exit("average_mode1")
		res=self.query(cmd)
		if res == "OK\n":pass
		else : 
			print("パワーセンサの平均化方式を設定できませんでした。シリアル通信タイムアウトの可能性があります。")
			sys.exit("average_mode2")
		return True
		
	def show_average_mode(self):
		cmd = "AVGTYP?\n"
		res = self.query(cmd)
		if res == "0\n":
			res = "Moving ( 移動法 )"
		elif res == "1\n":
			res = "Repeat ( 繰り返し法 )"
		else:
			print("パワーセンサの平均化方式を取得出来ませんでした。シリアル通信タイムアウトの可能性があります。")
			sys.exit("average_mode3")
		print("Average Mode = "+ res)
		return True	
		
	def set_average_count(self,avc=32):
		if avc >=1025:
			print("平均化回数の設定値が大きすぎます。引数 avc は 1024 以下の自然数で設定してください。")
			sys.exit("chapert1")
		elif avc < 1 :
			print("平均化回数の設定値 avc は 1024 以下の自然数で設定してください。")
		cmd = "AVGCNT %i\n"%avc
		res=self.query(cmd)
		if res == "OK\n":pass
		else : 
			print("平均化回数を設定できませんでした。シリアル通信タイムアウトの可能性があります。")
			sys.exit("avgcnt1")
		return True
		
	def show_average_count(self):
		cmd = "AVGCNT?\n"
		try:
			res = self.query(cmd).rstrip("\n")
			print("Average Count ( 平均化回数 ) = %s [回]"%res)
		except:
			print("平均化回数を取得できませんでした。シリアル通信タイムアウトの可能性があります。")
			sys.exit("avgcnt2")
		return True	
	
	def show_average(self):
		self.show_average_mode()
		self.show_average_count()
		return True
	
	def IP_meas(self,wt = 0.200, count = 999999 ):
		
		try:
			self.connect_Quick()
			self.ser.open()
		except:
			self.close()
			self.connect_Quick()
			self.ser.open()
		st = time.time()
		i=1

		cmd = "PWR?\n".encode()
		st = time.time()		
		
		try:
			while i <= count :
				ct = time.time()
				pt = ct-st
				self.ser.write(cmd)
				data = float(self.ser.read(100).decode())
				print (" Count = %d, Passtime = %.3f [s],Power = %.3f [dBm])"%(i, pt,data))
				i += 1
				delay2 = (i-1)*wt - (time.time()-st)
				time.sleep(delay2)
		finally: 
			self.close()
			self.connect()
		return True
		
	def IW_meas(self,wt = 0.500, count = 999999, filename = "Def" ):
	
		if filename == "Def":
			date = datetime.datetime.now()
			date = str(date).replace(":","-")
			filename = "PMdata-" +date[0:19] + ".txt"
		else: pass
		
		file = open(filename,"a")
		i=1
		file.close()

		try:
			self.connect_Quick()
			self.ser.open()
		except:
			self.close()
			self.connect_Quick()
			self.ser.open()

		cmd = "PWR?\n".encode()
		st = time.time()
		try:
			while i <= count :
				ct = time.time()
				pt = ct-st
				self.ser.write(cmd)
				data = float(self.ser.read(100).decode())
				file = open(filename,"a")
				file.write("Count = %d, Passtime = %.3f [s],Power = %.3f [dBm]\n"%(i, pt,data))
				file.close()
				i += 1
				delay2 = (i-1)*wt - (time.time()-st)
				time.sleep(delay2)
		finally: 
			self.ser.close()
			file.close()	
			self.connect()
		
		return True